In [1]:
#Import đầy đủ thư viện & Cố định Seed
import os
import time
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

# Import 2 Model + Baseline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

RANDOM_STATE = 42
DATA_PATH = "penguins_size.csv"
print("Đã import thư viện thành công!")

Đã import thư viện thành công!


In [2]:
!rm -rf /content/drive

In [3]:
#Kết nối google drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
print("Đã kết nối Google Drive!")

Mounted at /content/drive
Đã kết nối Google Drive!


In [4]:
# Tạo thư mục lưu model
DRIVE_DIR = "/content/drive/MyDrive/penhuin_models"
os.makedirs(DRIVE_DIR, exist_ok=True)
# File chỉ chưa model do phần 1 thực hiện
TRAINED_MODELS_PATH =os.path.join(
    DRIVE_DIR,
    "models_part1.joblib"
)
print("Thư mục lưu model:")
print(DRIVE_DIR)
print("\nFile model sẽ được lưu tại:")
print(TRAINED_MODELS_PATH)

Thư mục lưu model:
/content/drive/MyDrive/penhuin_models

File model sẽ được lưu tại:
/content/drive/MyDrive/penhuin_models/models_part1.joblib


In [5]:
#Load Dataset
df=pd.read_csv(DATA_PATH)
print("Kích thước dataset ban đầu:", df.shape)
print("\n5 dòng đầu:")
display(df.head())

Kích thước dataset ban đầu: (344, 7)

5 dòng đầu:


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [6]:
#Làm sạch dữ liệu cơ bản
#Trong dataset, một số giá trị sex có thể được ghi là "."
#Chuyển chúng thành NaN để xử lý bằng Imputer sau này.
df["sex"]=df["sex"].replace(".", np.nan)

#Các biến số
num_cols = [
    "culmen_length_mm",
    "culmen_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]
#Các biến phân loại
cat_cols = [
    "island",
    "sex"
]
#Biến mục tiêu
target_col = "species"

#Xóa các dòng không có bất kỳ dữ liệu số nào
df=df.dropna(
    subset=num_cols,
    how="all"
).reset_index(drop=True)

#Xóa dòng không có nhãn species
df=df.dropna(
    subset=[target_col]
).reset_index(drop=True)
print("\nKích thước dataset sau cleaning cơ bản:", df.shape)
print("\nMissing values:")
print(df[num_cols + cat_cols + [target_col]].isnull().sum())


Kích thước dataset sau cleaning cơ bản: (342, 7)

Missing values:
culmen_length_mm     0
culmen_depth_mm      0
flipper_length_mm    0
body_mass_g          0
island               0
sex                  9
species              0
dtype: int64


In [7]:
#Tách x và y
X = df[
    num_cols + cat_cols
]
y = df[target_col]

print("\nCác feature:")
print(X.columns.tolist())
print("\nCác class:")
print(y.unique())


Các feature:
['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g', 'island', 'sex']

Các class:
['Adelie' 'Chinstrap' 'Gentoo']


In [8]:
#Encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("\nMapping class:")
for class_name, encoded_value in zip(
    label_encoder.classes_,
    range(len(label_encoder.classes_))
):
  print(
    f"{class_name} -> {encoded_value}"
)


Mapping class:
Adelie -> 0
Chinstrap -> 1
Gentoo -> 2


In [9]:
#Chia train/ test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)
print("\nTrain/Test split:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


Train/Test split:
X_train: (273, 6)
X_test : (69, 6)


In [10]:
#Tạo preprocessor
#Pipeline xử lý biến số:
#Missing value -> median
#Sau đó StandardScaler
numeric_transformer = Pipeline(#
                               steps=[
                                   (
                                       "imputer",
                                       SimpleImputer(strategy="median")
                                   ),
                                   (
                                       "scaler",
                                       StandardScaler()
                                   )
                               ]
                               )
#Pipeline xử lý biến categorical:
#Missing value -> most frequent
#Sau đó OneHotEncoder
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

#Kết hợp hai pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            num_cols
        ),
        (
            "cat",
            categorical_transformer,
            cat_cols
        )
    ]
)
print("\nPreprocessor đã được tạo thành công!")


Preprocessor đã được tạo thành công!


In [11]:
#Cross validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)
#Dictionary lưu kết quả
results = {}
#Dictionary lưu pipeline tốt nhất
trained_pipelines={}

In [12]:
# HÀM TRAIN + GRID SEARCH
def train_and_tune(
    name,
    estimator,
    param_grid,
    scoring="f1_macro"
):

    # Tạo Pipeline:
    # Preprocessing -> Model
    pipe = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                estimator
            )
        ]
    )

    # GridSearchCV
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    # -----------------------------
    # TRAIN
    # -----------------------------

    t0 = time.time()

    grid.fit(
        X_train,
        y_train
    )

    train_time = time.time() - t0

    # -----------------------------
    # PREDICT
    # -----------------------------

    t0 = time.time()

    y_pred = grid.predict(
        X_test
    )

    predict_time = (
        time.time() - t0
    ) / len(X_test)

    # -----------------------------
    # LƯU BEST MODEL
    # -----------------------------

    trained_pipelines[name] = (
        grid.best_estimator_
    )

    # -----------------------------
    # LƯU KẾT QUẢ
    # -----------------------------

    results[name] = {
        "best_params": grid.best_params_,
        "cv_best_score": grid.best_score_,
        "train_time_sec": train_time,
        "predict_time_sec_per_sample": predict_time
    }

    # -----------------------------
    # HIỂN THỊ
    # -----------------------------

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(
        "Best params:",
        grid.best_params_
    )

    print(
        "CV best score (F1 Macro):",
        round(
            grid.best_score_,
            4
        )
    )

    print(
        f"Train time: {train_time:.3f}s"
    )

    print(
        f"Predict time/sample: "
        f"{predict_time:.6f}s"
    )

    return grid.best_estimator_


In [13]:
#Baseline Model
baseline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            DummyClassifier(
                strategy="most_frequent",
                random_state=RANDOM_STATE
            )
        )
    ]
)
baseline.fit(
    X_train,
    y_train
)
#Lưu Baseline
trained_pipelines["Baseline"]=baseline
#Đánh giá sơ bộ
acc_base = baseline.score(
    X_test,
    y_test
)
print("\n" + "=" *60)
print("BASELINE - DummyClassifier")
print("=" * 60)
print(
    f"Accuracy trên Test: {acc_base:.4f}"
)


BASELINE - DummyClassifier
Accuracy trên Test: 0.4348


In [14]:
# MODEL 1 - LOGISTIC REGRESSION

logreg_param_grid = {
    "classifier__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],

    "classifier__penalty": [
        "l2"
    ],

    "classifier__solver": [
        "lbfgs"
    ],

    "classifier__max_iter": [
        2000
    ]
}


model_logreg = train_and_tune(
    name="LogisticRegression",

    estimator=LogisticRegression(
        random_state=RANDOM_STATE
    ),

    param_grid=logreg_param_grid
)


LogisticRegression
Best params: {'classifier__C': 10, 'classifier__max_iter': 2000, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
CV best score (F1 Macro): 0.9912
Train time: 1.896s
Predict time/sample: 0.000077s


In [15]:
#Model 2: KNN
knn_param_grid = {
    "classifier__n_neighbors":[
        3,
        5,
        7,
        9,
        11,
        15
    ],
    "classifier__weights":[
        "uniform",
        "distance"
    ],
    "classifier__metric":[
        "euclidean",
        "manhattan"
    ]
}
model_knn = train_and_tune(
    name="KNN",
    estimator=KNeighborsClassifier(),
    param_grid=knn_param_grid
)


KNN
Best params: {'classifier__metric': 'manhattan', 'classifier__n_neighbors': 7, 'classifier__weights': 'distance'}
CV best score (F1 Macro): 0.9957
Train time: 1.604s
Predict time/sample: 0.000078s


In [16]:
#Kiểm tra các model đã train
print("\n" + "=" *60)
print("CÁC MODEL ĐÃ TRAIN")
print("="*60)
print(
    list(trained_pipelines.keys())
)


CÁC MODEL ĐÃ TRAIN
['Baseline', 'LogisticRegression', 'KNN']


In [17]:
#Tạo bảng tổng hợp training
summary_df = pd.DataFrame(
    results
).T
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
display(summary_df)


TRAINING SUMMARY


,best_params,cv_best_score,train_time_sec,predict_time_sec_per_sample
LogisticRegression,"{'classifier__C': 10, 'classifier__max_iter': ...",0.991162,1.896304,0.000077
KNN,"{'classifier__metric': 'manhattan', 'classifie...",0.995683,1.603982,0.000078


In [18]:
#Lưu training summary
SUMMARY_DIR="/content/drive/MyDrive/penguin_models"
os.makedirs(
    SUMMARY_DIR,
    exist_ok=True
)
summary_path=os.path.join(
    SUMMARY_DIR,
    "training_summary_part1.csv"
)
summary_df.to_csv(
    summary_path
)
print(summary_path)

/content/drive/MyDrive/penguin_models/training_summary_part1.csv


In [19]:
#Lưu các pipeline đã train
joblib.dump(
    trained_pipelines,
    TRAINED_MODELS_PATH
)
print("\n"+"="*60)
print("ĐÃ LƯU MODEL")
print("="*60)
print(
    "Đường dẫn:"
)
print(
    TRAINED_MODELS_PATH
)
print(
    "Số lượng pipeline:",
    len(trained_pipelines)
)
print(
    "Danh sách:",
    list(trained_pipelines.keys())
)


ĐÃ LƯU MODEL
Đường dẫn:
/content/drive/MyDrive/penhuin_models/models_part1.joblib
Số lượng pipeline: 3
Danh sách: ['Baseline', 'LogisticRegression', 'KNN']


In [20]:
#Kiểm tra file có thực sự tồn tại
file_exists=os.path.exists(
    TRAINED_MODELS_PATH
)
print("\nFile có tồn tại không:", file_exists)
if file_exists:
  file_size=os.path.getsize(
      TRAINED_MODELS_PATH
  )
  print(
      "Kích thước file:",
      file_size,
      "bytes"
  )


File có tồn tại không: True
Kích thước file: 57714 bytes


In [21]:
#Load thử lại model
test_loaded_models=joblib.load(
    TRAINED_MODELS_PATH
)
print("\n"+"="*60)
print("KIỂM TRA LOAD MODEL")
print("="*60)
print(
    "Đã load thành công!"
)
print(
    "Các model:",
    list(test_loaded_models.keys())
)


KIỂM TRA LOAD MODEL
Đã load thành công!
Các model: ['Baseline', 'LogisticRegression', 'KNN']


In [22]:
#Kết thúc notebook 3_train
print("\n"+"="*60)
print("HOÀN THÀNH 03_train")
print("="*60)
print(
    "Model của phần này:"
)
for model_name in trained_pipelines.keys():
  print(
      " -",
      model_name
  )
print(
    "\nRandom Forest và Gradient Boosting"
    "sẽ do thành viên khác thực hiện."
)


HOÀN THÀNH 03_train
Model của phần này:
 - Baseline
 - LogisticRegression
 - KNN

Random Forest và Gradient Boostingsẽ do thành viên khác thực hiện.


In [23]:
import os
import joblib

MODEL_DIR = "/content/drive/MyDrive/penguin_models/models"

os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "models_part1.joblib"
)

joblib.dump(
    trained_pipelines,
    MODEL_PATH
)

print("Đã lưu model tại:")
print(MODEL_PATH)
print("File tồn tại:", os.path.exists(MODEL_PATH))

Đã lưu model tại:
/content/drive/MyDrive/penguin_models/models/models_part1.joblib
File tồn tại: True


In [24]:
print(os.listdir("/content/drive/MyDrive/penguin_models/models"))

['models_part1.joblib']


In [25]:
import joblib
import os

# Lưu tạm vào runtime của Notebook 03
TEMP_MODEL = "/content/models_part1.joblib"

joblib.dump(trained_pipelines, TEMP_MODEL)

print("File tạm:", TEMP_MODEL)
print("Tồn tại:", os.path.exists(TEMP_MODEL))

File tạm: /content/models_part1.joblib
Tồn tại: True


In [26]:
!ls -lh /content/models_part1.joblib

-rw-r--r-- 1 root root 57K Sep 27 10:22 /content/models_part1.joblib


In [27]:
!ls -lh /content/models_part1.joblib

-rw-r--r-- 1 root root 57K Sep 27 10:22 /content/models_part1.joblib


In [28]:
from google.colab import files

files.download("/content/models_part1.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
print(type(trained_pipelines))
print(trained_pipelines)

<class 'dict'>
{'Baseline': Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['culmen_length_mm',
                                                   'culmen_depth_mm',
                                                   'flipper_length_mm',
                                                   'body_mass_g']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
            

In [30]:
print("TEMP_MODEL =", TEMP_MODEL)

TEMP_MODEL = /content/models_part1.joblib


In [31]:
import os

print("Tồn tại:", os.path.exists(TEMP_MODEL))

if os.path.exists(TEMP_MODEL):
    print("Kích thước:", os.path.getsize(TEMP_MODEL), "bytes")

Tồn tại: True
Kích thước: 57714 bytes


In [32]:
import joblib

loaded_models = joblib.load(TEMP_MODEL)

print(type(loaded_models))
print(loaded_models.keys())

<class 'dict'>
dict_keys(['Baseline', 'LogisticRegression', 'KNN'])


In [33]:
for name, pipeline in loaded_models.items():
    print(name)
    print("  Pipeline:", type(pipeline))
    print("  Classifier:", type(pipeline.named_steps["classifier"]))
    print()

Baseline
  Pipeline: <class 'sklearn.pipeline.Pipeline'>
  Classifier: <class 'sklearn.dummy.DummyClassifier'>

LogisticRegression
  Pipeline: <class 'sklearn.pipeline.Pipeline'>
  Classifier: <class 'sklearn.linear_model._logistic.LogisticRegression'>

KNN
  Pipeline: <class 'sklearn.pipeline.Pipeline'>
  Classifier: <class 'sklearn.neighbors._classification.KNeighborsClassifier'>



In [34]:
import os

print(os.path.exists("/content/models_part1.joblib"))
print(os.path.getsize("/content/models_part1.joblib"))

True
57714


In [35]:
import joblib

models = joblib.load("/content/models_part1.joblib")

print(models.keys())

for name, pipeline in models.items():
    print(name, "->", type(pipeline.named_steps["classifier"]))

dict_keys(['Baseline', 'LogisticRegression', 'KNN'])
Baseline -> <class 'sklearn.dummy.DummyClassifier'>
LogisticRegression -> <class 'sklearn.linear_model._logistic.LogisticRegression'>
KNN -> <class 'sklearn.neighbors._classification.KNeighborsClassifier'>


In [36]:
from google.colab import files

files.download("/content/models_part1.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
from sklearn.svm import SVC

In [39]:
# MODEL 3 - SUPPORT VECTOR MACHINE

svm_param_grid = {
    "classifier__C": [0.1, 1, 10],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma": ["scale", "auto"]
}

svm_pipeline = train_and_tune(
    name="SVM",
    estimator=SVC(probability=True, random_state=RANDOM_STATE),
    param_grid=svm_param_grid
)


SVM
Best params: {'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}
CV best score (F1 Macro): 0.9955
Train time: 3.529s
Predict time/sample: 0.000067s


In [40]:
from sklearn.ensemble import GradientBoostingClassifier

In [41]:
# MODEL 4 - GRADIENT BOOSTING

gb_param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_depth": [2, 3]
}

gb_pipeline = train_and_tune(
    name="Gradient Boosting",
    estimator=GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
    param_grid=gb_param_grid
)


Gradient Boosting
Best params: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 2, 'classifier__n_estimators': 100}
CV best score (F1 Macro): 0.9792
Train time: 7.442s
Predict time/sample: 0.000103s
